Phylogenetic signal for manuscript metadata
===========================================

Here we determine the historical covariates of the phylogenetic structure.

There are three traits: time, place, and institution (cursus) of origin.
All three have categorical codings, and time and place also have continuous coding.

Phylogenetic signal of categorical traits is measured with phylogenetic delta,
from Ribiero et al.'s Python implementation. This relies on an approximation of entropy
in ancestral states, so it is necessary to first run ancestral state reconstruction.
All this adapted from the notebook at
`https://github.com/diogo-s-ribeiro/delta-statistic/blob/master/Delta-Python/MANUAL.ipynb`
Note that ASR requires a rooted tree, but one that retains the branches as phylogenetic distance,
not just time. We use the pre-DTE rooted tree.

Phylogenetic signal of continuous traits is measured with Mantel's tests.
These are sometimes unreliable, but our task (actually comparing distances, phylogenetic
and geographic or temporal) doesn't have many other readily available statistical tools.
Phylogenetic signal of geographic distributions is usually only measured in latitude,
as proxy for climate, but we need actual distance.
Here we do not need the tree to be rooted, but since we just use distances on the tree,
the root never plays into this (the distance over the branch that was split by the root
remains the same), so we use the same tree as for the categorical traits to make the phylogenetic
distance more straightforwardly consistent across both categorical and continuous traits.


Additional libraries required to run the notebook
-------------------------------------------------

Bioinformatics and other specialised libraries:

- pastml
- Biopython
- skbio
- geopy

Generic data science:

- numpy
- pandas
- matplotlib



***TODO Rewrite for 08_phylogenetic_signal/ repo structure!** 

We use for `introits` data:

- `concatenated.characters.csv` as the traits file (to be replaced by a file with provenance, century, cursus, etc.)
- `tree4_alignment_and_trees.consensus.nwk` as the phylogenetic tree. This was obtained from `tree4` posterior consensus tree
  in `introits/04_divtime` through FigTree Newick export, because the extended Newick string from DTE results is not supported
  by PastML.

For `christmas` data:

- ...

Ancestral states estimation using PastML
-----------------------------------------

Setting up Ancestral state reconstruction as pre-requisite for phylogenetic delta.
Directly copied over from their notebook, just slightly reformatted to make arguments clearer.

In [2]:
import os
import time
import pandas as pd
import numpy as np
import math
import pprint
import pickle


from pastml.tree import read_tree, name_tree
from pastml.acr import acr
from pastml.annotation import preannotate_forest
from pastml import col_name2cat
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
import copy
def permute_dataframe_row_values(df):
    """This function is used to create metadata permutations 
    for checking the significance of phylogenetic delta."""
    index_values = copy.deepcopy(df.index.values)
    shuffled_df = df.sample(frac=1)
    shuffled_df.set_index(index_values, inplace=True)
    return shuffled_df

In [4]:
def _validate_input(tree_nwk, 
                    data, 
                    data_sep=',', 
                    single_tree_file=False,
                    shuffle_data_values=False):
    '''tree_nwk      = Represents the path to the Newick file containing the tree or a string with the tree itself.
    data             = Represents the path to the data file or DataFrame used for annotation with leaf states.
    data_sep         (default: ',')   = Separator used in the data file.
    single_tree_file (default: False) = Boolean value that specifies whether the input tree is provided as a single file.
    shuffle_data_values (defulat: False) = Randomly permute the data. Use this in order to test for significance 
                                           of phylogenetic delta vis-a-vis the dataset, controlling for the structure
                                           of the traits data itself. (Inspired by permutations in Mantel's test.)
    '''
    
    if single_tree_file==False:
        with open(tree_nwk, 'r') as f:                                                 # Reads the tree from a Newick file and returns its roots
            nwks = f.read().replace('\n', '')
        roots = [read_tree(tree_nwk)]
    else:
        roots = [read_tree(tree_nwk)]                                                  # Reads the newick tree and returns its roots

    column2annotated = Counter()                                                       # Counter to keep track of the number of times each column is annotated
    column2states    = defaultdict(set)                                                # Dictionary to store the unique states for each column

    # Read the data as a pandas DataFrame
    df         = pd.read_csv(data, sep=data_sep, index_col=0, header=0, dtype=str)
    if shuffle_data_values:
        df = permute_dataframe_row_values(df)
    
    df.index   = df.index.map(str)
    df.columns = [col_name2cat(column) for column in df.columns]
    columns    = df.columns    
    node_names     = set.union(*[{n.name for n in root.traverse() if n.name} for root in roots])     # Get the names of the nodes in the tree
    df_index_names = set(df.index)                                                                   # Get the index names from the DataFrame
    common_ids     = list(node_names & df_index_names)                                               # Find the common IDs between node names and DataFrame index names
    
    # strip quotes if needed
    if not common_ids:
        node_names = {_.strip("'").strip('"') for _ in node_names}
        common_ids = node_names & df_index_names
        if common_ids:
            for root in roots:
                for n in root.traverse():
                    n.name = n.name.strip("'").strip('"')

    # Preannotate the forest with the DataFrame
    preannotate_forest(roots, df=df)

    # Populate the column2states dictionary with unique states for each column
    for c in df.columns:
        column2states[c] |= {_ for _ in df[c].unique() if pd.notnull(_) and _ != ''}

    num_tips = 0

    # Count the number of annotated columns for each node
    column2annotated_states = defaultdict(set)
    for root in roots:
        for n in root.traverse():
            for c in columns:
                vs = getattr(n, c, set())
                column2states[c] |= vs
                column2annotated_states[c] |= vs
                if vs:
                    column2annotated[c] += 1
            if n.is_leaf():
                num_tips += 1

    if column2annotated:
        c, num_annotated = min(column2annotated.items(), key=lambda _: _[1])
    else:
        c, num_annotated = columns[0], 0

    # Calculate the percentage of unknown tip annotations
    percentage_unknown = (num_tips - num_annotated) / num_tips
    if percentage_unknown >= .9:
        raise ValueError('{:.1f}% of tip annotations for character "{}" are unknown, '
                         'not enough data to infer ancestral states. '
                         '{}'
                         .format(percentage_unknown * 100, c,
                                 'Check your annotation file and if its ids correspond to the tree tip/node names.'
                                 if data
                                 else 'You tree file should contain character state annotations, '
                                      'otherwise consider specifying a metadata file.'))
    c, states = min(column2annotated_states.items(), key=lambda _: len(_[1]))

    # Check if the number of unique states is too high for the given number of tips
    if len(states) > num_tips * .75:
        raise ValueError('Character "{}" has {} unique states annotated in this tree: {}, '
                         'which is too much to infer on a {} with only {} tips. '
                         'Make sure the character you are analysing is discrete, and if yes use a larger tree.'
                         .format(c, len(states), states, 'tree' if len(roots) == 1 else 'forest', num_tips))


    # Convert column2states to numpy arrays and sort the states
    column2states = {c: np.array(sorted(states)) for c, states in column2states.items()}

    # Name the trees in the forest
    for i, tree in enumerate(roots):
        name_tree(tree, suffix='' if len(roots) == 1 else '_{}'.format(i))

    return roots, columns, column2states, df


def marginal(tree, 
             data, 
             prediction_method='MPPA', 
             model='F81', 
             threads=0, 
             single_tree_file=False,
             return_everything=False,
             shuffle_data_values=False):
    '''tree           = Represents the path to the Newick file containing the tree or a string with the tree itself.
    data              = Represents the path to the data file or DataFrame used for annotation with leaf states.
    prediction_method (default: 'MPPA') = Specifies the ancestral character prediction method.
    model             (default: 'F81')  = Specifies the evolutionary model used for reconstruction.
    threads           (default: 0)      = Specifies the number of threads to use for the analysis.
    single_tree_file  (default: False)  = Boolean value that specifies whether the input tree is provided as a single file.'''

    # Set the number of threads based on the available CPU cores
    if threads < 1:
        threads = max(os.cpu_count(), 1)

    # Validate the input and get the roots, columns, and column2states
    roots, columns, column2states, df = \
        _validate_input(tree_nwk=tree, 
                        data=data, 
                        data_sep=',', 
                        single_tree_file=single_tree_file,
                        shuffle_data_values=shuffle_data_values)
    
    # DEBUG
    #print('Columns: {}'.format(columns))
    #print('Columns2states: {}'.format(column2states))
    
    # Perform the ancestral character reconstruction (ACR) analysis
    acr_results = acr(forest=roots, 
                      columns=columns, 
                      column2states=column2states, 
                      prediction_method=prediction_method, 
                      model=model, 
                      tau=0.01,   # Smoothing
                      threads=threads)

    # Get the leaf names from the tree
    leaf_names = read_tree(tree).get_leaf_names()

    # Get the marginal probabilities and exclude the leaf nodes
    # Note: acr_results[0] is just the first character!
    marginal   = np.asarray(acr_results[0]['marginal_probabilities'].drop(leaf_names))

    if return_everything:
        return marginal, acr_results, leaf_names, roots, columns, column2states, df
    
    return marginal

In [5]:
import logging

def delta_multicolumn(acr_results, 
                      df,
                      leaf_names, 
                      lambda0, se, sim, burn, thin, ent_type='LSE',):
    '''Computes phylogenetic signal delta for each position in the ACR results.
    Returns that as a dictionary. The key is 'character' value from each ACR result (trait),
    the value is the final delta value.'''
    deltas = {}
    state_counts = {}
    
    for position in acr_results:
        character = position['character']
        
        n_states = len(position['states'])
        state_counts[character] = n_states
        
        is_segregating = False
        if n_states > 1:
            is_segregating = True
        if not is_segregating:
            logging.debug('Character {} is not segregating! Skipping.'.format(character))
            deltas[character] = 0
            continue
                        
        # Is it informative?
        if n_states == 2:
            pass
            
        marginal_probs = np.asarray(position['marginal_probabilities'].drop(leaf_names))
        
        position_delta = delta(x=marginal_probs,
                               lambda0=lambda0, se=se, sim=sim, burn=burn, thin=thin, ent_type=ent_type)
        deltas[character] = position_delta
    return deltas, state_counts

Delta computation
-----------------

First we make sure we have the delta statistic code where it should be:

In [6]:
import importlib
import sys

if not os.path.isdir('delta-statistic'):
    !git clone https://github.com/diogo-s-ribeiro/delta-statistic.git

sys.path.append('delta-statistic/Delta-Python/')
from delta_functs import delta

Cloning into 'delta-statistic'...
remote: Enumerating objects: 194, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 194 (delta 66), reused 163 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (194/194), 18.59 MiB | 10.52 MiB/s, done.
Resolving deltas: 100% (66/66), done.


These are good default parameters for the delta computation.

In [7]:
lambda0  = 0.1                       # rate parameter of the proposal
se       = 0.5                       # standard deviation of the proposal
sim      = 100000                    # number of iterations
thin     = 10                        # Keep only each xth iterate
burn     = 100                       # Burned-in iterates

ent_type = 'LSE'                     # Linear Shannon Entropy

In [8]:
method     = "MPPA"                # MPPA, MAP
model      = "F81"                 # F81, JC, EFT

In [9]:
def compute_test_statistics_from_permuted_deltas(empirical_deltas, permuted_deltas, state_counts):
    proportion_permuted_greater = {}
    quantiles_005 = {}
    quantiles_001 = {}
    
    n_permutations = len(permuted_deltas)
    n_quantile_005 = max(int(n_permutations / 20), 1)  # Probably can move quantile sizes here, they are shared across characters.
    n_quantile_001 = max(int(n_permutations / 100), 1)
    
    if n_permutations < 100:
        logging.warning('Cannot safely determine quantiles from less than 100 permutations!')

    for c in empirical_deltas:
        #print('Checking c = {}'.format(c))
        if state_counts[c] <= 1:
            continue
        empirical_delta = empirical_deltas[c]
        
        permuted_deltas_c = [_p_deltas[c] for _p_deltas in permuted_deltas]
        n_larger_deltas = len([_p_d for _p_d in permuted_deltas_c if _p_d >= empirical_delta])
        proportion_permuted_greater[c] = n_larger_deltas / n_permutations

        # Determine qunatile values.
        sorted_permuted_deltas_c = sorted(permuted_deltas_c)
        quantile_005 = sorted_permuted_deltas_c[-n_quantile_005] 
        quantiles_005[c] = quantile_005
        quantile_001 = sorted_permuted_deltas_c[-n_quantile_001]
        quantiles_001[c] = quantile_001

    return proportion_permuted_greater, quantiles_005, quantiles_001

In [10]:
def run_experiment(path_tree, 
                   path_data, 
                   permutations=False,
                   accumulator_deltas=None,
                   accumulator_permuted_deltas=None):
    ancest_prob, acr_results, leaf_names, roots, columns, column2states, df = marginal(path_tree, 
                                                                                       path_data,
                                                                                       return_everything=True)
    deltas, state_counts = delta_multicolumn(acr_results, 
                                             df,
                                             leaf_names, 
                                             lambda0=lambda0, se=se, sim=sim, burn=burn, thin=thin, ent_type='LSE')   
    log_deltas = {c: math.log(1 + deltas[c]) for c in deltas}    

    # Accumulate empirical deltas. Side effect!
    if accumulator_deltas is not None:
        accumulator_deltas.append(deltas)
    
    print('Finished computing empirical deltas.')
    #print(deltas)
    
    # Get deltas distributions over permuted taxon data.
    if not permutations:
        permutations = 0

    permuted_deltas = []
    permuted_log_deltas = []
    _permutation_report_n = 10

    _start_time = time.time()
    for i in range(permutations):
        
        # Report every few permutations, just to keep track how far along the process is (it's slow).
        if ((i + 1) % _permutation_report_n == 0):
            _permutation_time = time.time()
            _runtime = _permutation_time - _start_time
            print('Permutation no. {}: Total time: {:.1f} s, {:.2f} s per permutation'.format(i + 1, _runtime, _runtime / i))

        # Now we run this with randomly permuted data values.
        # We put this in a try block because sometimes -- rarely, but sometimes -- some numeric issues
        # cause a crash in the marginal computation. In that case, we just skip the permutation: it is rare enough
        # to not affect anything.
        try:
            ancest_prob, acr_results, leaf_names, roots, columns, column2states, df = marginal(path_tree, 
                                                                                               path_data,
                                                                                               return_everything=True,
                                                                                               shuffle_data_values=True)
        except ValueError as e:
            print('Skipped permutation no. {}: numerical error encountered in marginal computation.'.format(i + 1))
            continue
            
        _p_deltas, _p_state_counts = delta_multicolumn(acr_results, 
                                                       df,
                                                       leaf_names, 
                                                       lambda0=lambda0, se=se, sim=sim, burn=burn, thin=thin, ent_type='LSE')   
        _p_log_deltas = {c: math.log(1 + deltas[c]) for c in deltas} 
        permuted_deltas.append(_p_deltas)
        permuted_log_deltas.append(_p_log_deltas)
        
        # Accumulate deltas for R0 of no phylogenetic signal. Side effect!
        if accumulator_permuted_deltas is not None:
            accumulator_permuted_deltas.append(_p_deltas)

    _end_time = time.time()
    _runtime = _end_time - _start_time
    print('Total time: {:.1f} s, {:.2f} s per permutation'.format(_runtime, _runtime / (N_PERMUTATIONS + 1)))        
        
    # Can re-use state counts here, because we are only permuting rows, not pertrubing values or boostrapping columns or some such.
    proportion_permuted_greater, quantiles_005, quantiles_001 = compute_test_statistics_from_permuted_deltas(deltas, 
                                                                                                             permuted_deltas,
                                                                                                             state_counts=state_counts)

    return deltas, log_deltas, permuted_deltas, state_counts, proportion_permuted_greater, quantiles_005, quantiles_001

In [11]:
def report_experiment(path_tree, path_data,
                      deltas, log_deltas, state_counts, 
                      proportion_permuted_greater=None, quantiles_005=None, quantiles_001=None, 
                      permutations=False,
                      as_csv=False):

    if as_csv:
        header = 'character,n_states,log_delta,delta,p,q005,q001'
        print(header)
        for c in deltas:
            if state_counts[c] > 1:
                output_string = ','.join([str(c), str(state_counts[c]), 
                                          '{:.3f}'.format(log_deltas[c]), 
                                          '{:.3f}'.format(deltas[c]), 
                                          '{:.3f}'.format(proportion_permuted_greater[c]), 
                                          '{:.3f}'.format(quantiles_005[c]), 
                                          '{:.3f}'.format(quantiles_001[c])])
                print(output_string)
        return
    
    # Reporting
    print('')
    print('Phylogenetic delta')
    print('==================')
    print('Tree: {}'.format(path_tree))
    print('Data: {}'.format(path_data))
    print('\nTraits:')
    print('-------')
    for c in deltas:
        if state_counts[c] > 1:
            output_string = '{}:\t{} states \tlog(1+d)={:.3f}  \td={:.3f} '.format(c, state_counts[c], log_deltas[c], deltas[c])
            if permutations:
                output_string += '\t (p={:.3f}, q005={:.3f}, q001={:.3f})'.format(proportion_permuted_greater[c], quantiles_005[c], quantiles_001[c])
            print(output_string)
            

Results: Pylogenetic delta in the traits
----------------------------------------

Categorical metadata

In [13]:
#path_data  = r"adorate.characters.csv"  # File containing tip/node annotations, in csv or tab format
path_data  = r"../data/christmas-metadata.csv"
#path_tree  = r"introits.tree4.consensus.nwk"      # File containing Newick tree -- rooted, for delta phylogenetic signal analysis
#path_tree  = r"introits.unrooted.nwk"
# We should use the rooted tree *before* DTE is computed!
path_tree = r"../data/christmas.concatenated.rooted-predte.nwk"

**Christmas**

Including bootstrap (permuted trait values), to check for significance.
Each permutation takes approx. 1.22 s on a good laptop, because ancestral state reconstruction needs to be re-run with the permuted trait values, so expect the 1000 iterations to run for a while.

In [14]:
N_PERMUTATIONS = 10 # For testing the notebook. Actual experiment results should be computed with 1000 permutations or more.

In [15]:
# Condition: 
#   - categorical traits
#   - christmas dataset
#   - including Cistercians

PATH_DATA = r"../data/christmas-metadata.csv"
PATH_TREE = r"../data/christmas.concatenated.rooted-predte.nwk"

# Christmas, INCLUDING Cistercians
deltas, log_deltas, permuted_deltas, state_counts, proportions_permuted_greater, q005, q001 = run_experiment(
    path_tree=PATH_TREE, 
    path_data=PATH_DATA,
    permutations=N_PERMUTATIONS)

Finished computing empirical deltas.
Permutation no. 10: Total time: 10.8 s, 1.20 s per permutation


Total time: 12.0 s, 1.09 s per permutation


In [16]:
report_experiment(PATH_TREE, PATH_DATA, 
                  deltas, log_deltas, state_counts, proportions_permuted_greater, q005, q001, 
                  permutations=N_PERMUTATIONS)


Phylogenetic delta
Tree: ../data/christmas.concatenated.rooted-predte.nwk
Data: ../data/christmas-metadata.csv

Traits:
-------
century:	6 states 	log(1+d)=0.341  	d=0.407 	 (p=0.700, q005=0.605, q001=0.605)
provCoarse:	5 states 	log(1+d)=1.081  	d=1.948 	 (p=0.000, q005=0.704, q001=0.704)
provenanceFine:	5 states 	log(1+d)=0.890  	d=1.434 	 (p=0.000, q005=0.877, q001=0.877)
cursusDetailed:	5 states 	log(1+d)=1.091  	d=1.976 	 (p=0.000, q005=0.620, q001=0.620)
cursusCoarse:	2 states 	log(1+d)=0.833  	d=1.301 	 (p=0.100, q005=3.125, q001=3.125)
proveEWCist:	3 states 	log(1+d)=2.088  	d=7.065 	 (p=0.000, q005=0.959, q001=0.959)
provEW:	2 states 	log(1+d)=2.592  	d=12.355 	 (p=0.000, q005=1.544, q001=1.544)


**Christmas, excluding Cistercians**

In [17]:
# Condition: 
#   - categorical traits
#   - christmas dataset
#   - no Cistercians

PATH_DATA = r"../data/christmas-noncist-metadata.csv"
PATH_TREE = r"../data/christmas.concatenated.rooted-predte.nwk"

# Christmas, EXCLUDING Cistercians
deltas_nC, log_deltas_nC, permuted_deltas_nC, state_counts_nC, p_nC, q005_nC, q001_nC = run_experiment(
    path_tree=PATH_TREE, 
    path_data=PATH_DATA,
    permutations=N_PERMUTATIONS)

Finished computing empirical deltas.
Permutation no. 10: Total time: 10.6 s, 1.18 s per permutation


Total time: 11.8 s, 1.08 s per permutation


In [18]:
report_experiment(PATH_TREE, PATH_DATA, deltas_nC, log_deltas_nC, state_counts_nC, p_nC, q005_nC, q001_nC, permutations=N_PERMUTATIONS)


Phylogenetic delta
Tree: ../data/christmas.concatenated.rooted-predte.nwk
Data: ../data/christmas-noncist-metadata.csv

Traits:
-------
century:	5 states 	log(1+d)=0.524  	d=0.690 	 (p=1.000, q005=1.491, q001=1.491)
provCoarse:	4 states 	log(1+d)=1.471  	d=3.355 	 (p=0.000, q005=0.642, q001=0.642)
provenanceFine:	5 states 	log(1+d)=1.105  	d=2.019 	 (p=0.000, q005=0.395, q001=0.395)
cursusDetailed:	4 states 	log(1+d)=0.708  	d=1.031 	 (p=0.100, q005=1.555, q001=1.555)
cursusCoarse:	2 states 	log(1+d)=0.695  	d=1.004 	 (p=0.100, q005=1.043, q001=1.043)
proveEWCist:	2 states 	log(1+d)=1.458  	d=3.295 	 (p=0.000, q005=2.182, q001=2.182)
provEW:	2 states 	log(1+d)=2.229  	d=8.289 	 (p=0.000, q005=0.872, q001=0.872)


**Introits** 

(Including bootstrap)

In [20]:
# Condition: 
#   - categorical traits
#   - introits dataset
#   - including Cistercians

PATH_DATA = r"../data/introits-metadata.csv"
PATH_TREE = r"../data/introits.tree4.pre-dte.nwk"

# Christmas, INCLUDING Cistercians
i_deltas, i_log_deltas, i_permuted_deltas, i_state_counts, i_proportions_permuted_greater, i_q005, i_q001 = run_experiment(
    path_tree=PATH_TREE, 
    path_data=PATH_DATA,
    permutations=N_PERMUTATIONS)

Finished computing empirical deltas.
Permutation no. 10: Total time: 17.3 s, 1.93 s per permutation
Permutation no. 20: Total time: 35.3 s, 1.86 s per permutation
Permutation no. 30: Total time: 53.8 s, 1.86 s per permutation


Total time: 56.1 s, 1.81 s per permutation


In [21]:
report_experiment(PATH_TREE, PATH_DATA, i_deltas, i_log_deltas, i_state_counts, i_proportions_permuted_greater, i_q005, i_q001, permutations=N_PERMUTATIONS)


Phylogenetic delta
Tree: ../data/introits.tree4.pre-dte.nwk
Data: ../data/introits-metadata.csv

Traits:
-------
century:	4 states 	log(1+d)=0.679  	d=0.971 	 (p=0.733, q005=2.675, q001=2.675)
provCoarse:	7 states 	log(1+d)=1.272  	d=2.570 	 (p=0.000, q005=1.777, q001=1.777)
provenanceFine:	8 states 	log(1+d)=1.384  	d=2.990 	 (p=0.000, q005=0.979, q001=0.979)
cursusDetailed:	7 states 	log(1+d)=0.986  	d=1.682 	 (p=0.000, q005=1.157, q001=1.157)
cursusCoarse:	2 states 	log(1+d)=0.392  	d=0.480 	 (p=0.367, q005=2.998, q001=2.998)
provEWSCist:	4 states 	log(1+d)=3.017  	d=19.436 	 (p=0.000, q005=6.850, q001=6.850)
provEWS:	4 states 	log(1+d)=2.942  	d=17.949 	 (p=0.000, q005=2.630, q001=2.630)


In [23]:
# Condition: 
#   - categorical traits
#   - introits dataset
#   - no Cistercians

PATH_DATA = r"../data/introits-noncist-metadata.csv"
PATH_TREE = r"../data/introits.tree4.pre-dte.nwk"

# Christmas, INCLUDING Cistercians
i_deltas_nC, i_log_deltas_nC, i_permuted_deltas_nC, i_state_counts_nC, i_proportions_permuted_greater_nC, i_q005_nC, i_q001_nC = run_experiment(
    path_tree=PATH_TREE, 
    path_data=PATH_DATA,
    permutations=N_PERMUTATIONS)

Finished computing empirical deltas.
Permutation no. 10: Total time: 13.3 s, 1.48 s per permutation
Permutation no. 20: Total time: 28.4 s, 1.49 s per permutation
Permutation no. 30: Total time: 43.8 s, 1.51 s per permutation


Total time: 45.2 s, 1.46 s per permutation


In [24]:
report_experiment(PATH_TREE, PATH_DATA, 
                  i_deltas_nC, i_log_deltas_nC, i_state_counts_nC, 
                  i_proportions_permuted_greater_nC, i_q005_nC, i_q001_nC, permutations=N_PERMUTATIONS)


Phylogenetic delta
Tree: ../data/introits.tree4.pre-dte.nwk
Data: ../data/introits-noncist-metadata.csv

Traits:
-------
century:	4 states 	log(1+d)=0.824  	d=1.280 	 (p=0.067, q005=1.334, q001=1.334)
provCoarse:	6 states 	log(1+d)=1.367  	d=2.924 	 (p=0.000, q005=2.803, q001=2.803)
provenanceFine:	8 states 	log(1+d)=1.284  	d=2.610 	 (p=0.000, q005=0.731, q001=0.731)
cursusDetailed:	6 states 	log(1+d)=0.322  	d=0.380 	 (p=0.333, q005=1.182, q001=1.182)
cursusCoarse:	2 states 	log(1+d)=0.012  	d=0.012 	 (p=0.933, q005=1.643, q001=1.643)
provEastWest:	3 states 	log(1+d)=2.321  	d=9.190 	 (p=0.000, q005=2.671, q001=2.671)


Continuous traits
==============================

In the continuos condition, we'd like to measure the correlation between phylogenetic distance and physical distance.


In [25]:
import geopy
from geopy import distance
from Bio import Phylo
import csv

# TODO: Mantel test: is correlation significant when we permute the distances?
# Note that it is potentially problematic: https://besjournals.onlinelibrary.wiley.com/doi/full/10.1111/2041-210X.12425
import skbio
import numpy as np

In [26]:
# Combination of both Introits and Christmas Cistercian sources.
# They are filtered with "not in CISTERCIAN_SOURCES", so this does not cause issues.
CISTERCIAN_SOURCES = ['_CH_ROM_Ms_liturg_FiD_5',
                      '_D_HEu_Cod_Sal_X_007',
                      '_F_Pn_NAL_01414',
                      '_PL_WRu_I_F_414',
                      '_PL_WRu_I_F_416'] + [
    'A_Wn_1799',
    'CDN_Hsmu_M2149_L4',
    'D_KNd_1161'
]


In [27]:
# These are experiment-specific settings.
path_latlong_data = 'christmas_inputs/christmas-lat-long.csv'
path_chrono_data = 'christmas_inputs/christmas-dates.csv'
path_tree = 'christmas_inputs/christmas.concatenated.rooted-predte.nwk'

EXCLUDE_CISTERCIANS = True

In [28]:
def load_traits_continuous_data(path_latlong_data, path_chrono_data):
    # Load latitude/logitude data
    latlong_data = {}
    with open(path_latlong_data) as fh:
        reader = csv.DictReader(fh, delimiter=',')
        for row in reader:
            latlong_data[row['tip_ID']] = (float(row['lat']), float(row['long']))

    # Compute geographic distances.
    geo_pairwise_distances = {}
    for m in latlong_data:
        geo_pairwise_distances[m] = {}
        for n in latlong_data:
            geo_dist = distance.distance(latlong_data[m], latlong_data[n]).kilometers
            geo_pairwise_distances[m][n] = geo_dist

    # Load chronology data.
    chrono_data = {}
    with open(path_chrono_data) as fh:
        reader = csv.DictReader(fh, delimiter=',')
        for row in reader:
            chrono_data[row['tip_ID']] = (float(row['post-quam']), float(row['ante-quem']))

    # Compute min, max, and average chronological distances.
    chrono_pairwise_distances_shortest = {}
    chrono_pairwise_distances_longest = {}
    chrono_pairwise_distances_mean = {}
    for m in chrono_data:
        chrono_pairwise_distances_shortest[m] = {}
        chrono_pairwise_distances_longest[m] = {}
        chrono_pairwise_distances_mean[m] = {}
        m_a, m_p = chrono_data[m]
        m_avg = (m_p + m_a) / 2
        for n in chrono_data:
            n_a, n_p = chrono_data[n]
            n_avg = (n_p + n_a) / 2
            d_min = min(abs(m_a - n_a), abs(m_p - n_p))
            chrono_pairwise_distances_shortest[m][n] = d_min
            d_max = max(abs(m_a - n_p), abs(m_p - n_a))
            chrono_pairwise_distances_longest[m][n] = d_max
            d_avg = abs(m_avg - n_avg)
            chrono_pairwise_distances_mean[m][n] = d_avg

    return latlong_data, chrono_data, \
           geo_pairwise_distances, \
           chrono_pairwise_distances_shortest, chrono_pairwise_distances_longest, chrono_pairwise_distances_mean
    # return {'latlong_data': latlong_data, 
    #         'geo_pairwise_distances': geo_pairwise_distances, 
    #         'chrono_data': chrono_data,
    #         'chrono_pairwise_distances_shortest': chrono_pairwise_distances_shortest,
    #         'chrono_pairwise_distances_longest': chrono_pairwise_distances_longest,
    #         'chrono_pairwise_distances_mean': chrono_pairwise_distances_mean}

In [29]:
# Here we use the unrooted tree, because we don't need ASR and logically, this analysis comes in *before* we do DTE.

def load_tree_and_phylo_distances(path_tree, latlong_data):
    #tree = Phylo.read('tree4_alignment_and_trees.consensus.nwk', format='newick')
    path_tree_geosignal = path_tree #r"introits_inputs/introits.tree4.pre-dte.nwk"
    tree = Phylo.read(path_tree_geosignal, format='newick')
    
    phylo_pairwise_distances = {}
    for m in latlong_data:
        phylo_pairwise_distances[m] = {}
        for n in latlong_data:
            phylo_dist = tree.distance(m, n)
            phylo_pairwise_distances[m][n] = phylo_dist

    return tree, phylo_pairwise_distances

In [30]:
# Build distance matrices.

def build_distance_matrices(latlong_data, # for tip ordering
                            phylo_pairwise_distances, 
                            geo_pairwise_distances,
                            chrono_pairwise_distances_shortest, 
                            chrono_pairwise_distances_longest, 
                            chrono_pairwise_distances_mean,
                            _exclude_cistercians):

    # Requires a constant ordering of tips, so that they are comparable!
    tip_ordering = sorted(latlong_data.keys())
    if _exclude_cistercians:
        tip_ordering = sorted([k for k in latlong_data.keys() if k not in CISTERCIAN_SOURCES])
    n_tips = len(tip_ordering)

    phylo_dist_matrix = np.zeros(shape=(n_tips, n_tips))
    geo_dist_matrix = np.zeros(shape=(n_tips, n_tips))
    chrono_min_dist_matrix = np.zeros(shape=(n_tips, n_tips))
    chrono_max_dist_matrix = np.zeros(shape=(n_tips, n_tips))
    chrono_avg_dist_matrix = np.zeros(shape=(n_tips, n_tips))

    for i in range(n_tips):
        for j in range(n_tips):
            if i == j:
                continue

            m, n = tip_ordering[i], tip_ordering[j]        

            # Phylogenetic distance
            phylo_dist = phylo_pairwise_distances[m][n]
            phylo_dist_matrix[i,j] = phylo_dist

            # Geographic distance
            geo_dist = geo_pairwise_distances[m][n]
            geo_dist_matrix[i,j] = geo_dist

            # Temporal distance
            chrono_min_dist = chrono_pairwise_distances_shortest[m][n]
            chrono_min_dist_matrix[i,j] = chrono_min_dist
            chrono_max_dist = chrono_pairwise_distances_longest[m][n]
            chrono_max_dist_matrix[i,j] = chrono_max_dist     
            chrono_avg_dist = chrono_pairwise_distances_mean[m][n]
            chrono_avg_dist_matrix[i,j] = chrono_avg_dist
        
    return phylo_dist_matrix, geo_dist_matrix, chrono_min_dist_matrix, chrono_max_dist_matrix, chrono_avg_dist_matrix, tip_ordering

In [31]:
# Note: we keep EXCLUDE_CISTERCIANS as a global setting for now.
#       Should later be refactored as some excluded_tips parameter here.
def run_mantel_test(path_latlong_data, path_chrono_data, path_tree, _exclude_cistercians=EXCLUDE_CISTERCIANS):
    
    # Load data and compute distances.
    latlong_data, chrono_data, geo_pairwise_distances, chrono_pairwise_distances_shortest, chrono_pairwise_distances_longest, chrono_pairwise_distances_mean = load_traits_continuous_data(
        path_latlong_data, 
        path_chrono_data)
    tree, phylo_pairwise_distances = load_tree_and_phylo_distances(path_tree, latlong_data=latlong_data)
    
    # Reformat as distance matrices.
    phylo_dist_matrix, geo_dist_matrix, chrono_min_dist_matrix, chrono_max_dist_matrix, chrono_avg_dist_matrix, tip_ordering = build_distance_matrices(
        latlong_data,
        phylo_pairwise_distances, 
        geo_pairwise_distances,
        chrono_pairwise_distances_shortest, 
        chrono_pairwise_distances_longest, 
        chrono_pairwise_distances_mean,
        _exclude_cistercians=_exclude_cistercians)

    
    # Run Mantel tests.
    print('Mantel tests results')
    print('--------------------')
    print('n. taxa: {}'.format(len(tip_ordering)))
    print('Cistercians excluded: {}'.format(_exclude_cistercians))
    
    _reject_string = '(!!)'
    _rejection_alpha = 0.05
    
    # Provenance
    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, geo_dist_matrix,
                                                    method='pearson',
                                                    #alternative='greater',  # We don't care about the other tail.
                                                    permutations=10000)
    print('GEO       | Pearson:  coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else '')) 
    
    # Report.
    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, geo_dist_matrix,
                                                    method='spearman',
                                                    #alternative='greater',  # We don't care about the other tail.
                                                    permutations=10000) 
    print('            Spearman: coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else '')) 
    
    
    # Time
    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, chrono_avg_dist_matrix,
                                                        method='pearson',
                                                        #alternative='greater',  # We don't care about the other tail.
                                                        permutations=10000) 
    print('CHRONOavg | Pearson:  coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else ''))     
    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, chrono_avg_dist_matrix,
                                                        method='spearman',
                                                        #alternative='greater',  # We don't care about the other tail.
                                                        permutations=10000) 
    print('            Spearman: coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else ''))     

    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, chrono_min_dist_matrix,
                                                        method='pearson',
                                                        #alternative='greater',  # We don't care about the other tail.
                                                        permutations=10000) 
    print('CHRONOmin | Pearson:  coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else '')) 
    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, chrono_min_dist_matrix,
                                                        method='spearman',
                                                        #alternative='greater',  # We don't care about the other tail.
                                                        permutations=10000) 
    print('            Spearman: coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else '')) 

    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, chrono_max_dist_matrix,
                                                        method='pearson',
                                                        #alternative='greater',  # We don't care about the other tail.
                                                        permutations=10000) 
    print('CHRONOmax | Pearson:  coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else '')) 
    coef, p_value, n_taxa = skbio.stats.distance.mantel(phylo_dist_matrix, chrono_max_dist_matrix,
                                                        method='spearman',
                                                        #alternative='greater',  # We don't care about the other tail.
                                                        permutations=10000) 
    print('            Spearman: coef.={:.3f}, p={:.4f} {}'.format(coef, p_value, _reject_string if p_value >= _rejection_alpha else '')) 

    
    print('')
    autocorr_coef, autocorr_p_value, autocorr_n_taxa = skbio.stats.distance.mantel(chrono_avg_dist_matrix, geo_dist_matrix,
                                                    method='pearson',
                                                    #alternative='greater',  # We don't care about the other tail.
                                                    permutations=10000) 
    print('AUTOCORR  | Pearson:  coef.={:.3f}, p={:.4f} {}'.format(autocorr_coef, autocorr_p_value, _reject_string if autocorr_p_value >= _rejection_alpha else '')) 
    autocorr_coef, autocorr_p_value, autocorr_n_taxa = skbio.stats.distance.mantel(chrono_avg_dist_matrix, geo_dist_matrix,
                                                    method='spearman',
                                                    #alternative='greater',  # We don't care about the other tail.
                                                    permutations=10000) 
    print('            Spearman: coef.={:.3f}, p={:.4f} {}'.format(autocorr_coef, autocorr_p_value, _reject_string if autocorr_p_value >= _rejection_alpha else '')) 

In [34]:
CHRISTMAS_PATHS = {'path_latlong_data': '../data/christmas-lat-long.csv',
                   'path_chrono_data': '../data/christmas-dates.csv',
                   'path_tree': '../data/christmas.concatenated.rooted-predte.nwk'}
INTROITS_PATHS = {'path_latlong_data': '../data/introits-lat-long.csv',
                  'path_chrono_data': '../data/introits-dates.csv',
                  'path_tree': '../data/introits.tree4.pre-dte.nwk'}

In [35]:
print('Continuous traits, Christmas dataset, including Cistercians:\n')
CHRISTMAS_PATHS['_exclude_cistercians'] = False
run_mantel_test(**CHRISTMAS_PATHS)

Continuous traits, Christmas dataset:

Mantel tests results
--------------------
n. taxa: 14
Cistercians excluded: False
GEO       | Pearson:  coef.=0.480, p=0.0029 
            Spearman: coef.=0.454, p=0.0043 
CHRONOavg | Pearson:  coef.=0.213, p=0.0327 
            Spearman: coef.=0.265, p=0.0199 
CHRONOmin | Pearson:  coef.=0.202, p=0.0371 
            Spearman: coef.=0.266, p=0.0180 
CHRONOmax | Pearson:  coef.=0.248, p=0.0245 
            Spearman: coef.=0.261, p=0.0223 

AUTOCORR  | Pearson:  coef.=0.171, p=0.1062 (!!)
            Spearman: coef.=0.170, p=0.0933 (!!)


In [36]:
print('Continuous traits, Christmas dataset, no Cistercians:\n')
CHRISTMAS_PATHS['_exclude_cistercians'] = True
run_mantel_test(**CHRISTMAS_PATHS)

Continuous traits, Christmas dataset:

Mantel tests results
--------------------
n. taxa: 11
Cistercians excluded: True
GEO       | Pearson:  coef.=0.861, p=0.0001 
            Spearman: coef.=0.844, p=0.0001 
CHRONOavg | Pearson:  coef.=0.411, p=0.0278 
            Spearman: coef.=0.415, p=0.0310 
CHRONOmin | Pearson:  coef.=0.402, p=0.0314 
            Spearman: coef.=0.403, p=0.0365 
CHRONOmax | Pearson:  coef.=0.454, p=0.0170 
            Spearman: coef.=0.460, p=0.0207 

AUTOCORR  | Pearson:  coef.=0.420, p=0.0144 
            Spearman: coef.=0.374, p=0.0288 


In [37]:
print('Continuous traits, Introits dataset, including Cistercians:\n')
INTROITS_PATHS['_exclude_cistercians'] = False
run_mantel_test(**INTROITS_PATHS)

Continuous traits, Introits dataset, including Cistercians:

Mantel tests results
--------------------
n. taxa: 23
Cistercians excluded: False
GEO       | Pearson:  coef.=0.316, p=0.0030 
            Spearman: coef.=0.323, p=0.0032 
CHRONOavg | Pearson:  coef.=-0.033, p=0.8082 (!!)
            Spearman: coef.=-0.015, p=0.9029 (!!)
CHRONOmin | Pearson:  coef.=-0.042, p=0.7557 (!!)
            Spearman: coef.=-0.025, p=0.8351 (!!)
CHRONOmax | Pearson:  coef.=0.019, p=0.8846 (!!)
            Spearman: coef.=0.067, p=0.5930 (!!)

AUTOCORR  | Pearson:  coef.=0.136, p=0.1129 (!!)
            Spearman: coef.=0.134, p=0.0784 (!!)


In [38]:
print('Continuous traits, Introits dataset, no Cistercians:\n')
INTROITS_PATHS['_exclude_cistercians'] = True
run_mantel_test(**INTROITS_PATHS)

Continuous traits, Introits dataset, no Cistercians:

Mantel tests results
--------------------
n. taxa: 18
Cistercians excluded: True
GEO       | Pearson:  coef.=0.582, p=0.0002 
            Spearman: coef.=0.639, p=0.0001 
CHRONOavg | Pearson:  coef.=-0.100, p=0.4413 (!!)
            Spearman: coef.=-0.058, p=0.6216 (!!)
CHRONOmin | Pearson:  coef.=-0.126, p=0.3470 (!!)
            Spearman: coef.=-0.060, p=0.6242 (!!)
CHRONOmax | Pearson:  coef.=-0.016, p=0.9066 (!!)
            Spearman: coef.=0.041, p=0.7385 (!!)

AUTOCORR  | Pearson:  coef.=0.080, p=0.4624 (!!)
            Spearman: coef.=0.055, p=0.6096 (!!)
